# Explainable AI (Vision) - Grad-CAM

This notebook demonstrates how to visualize *why* a computer vision model made a specific decision. We use **Grad-CAM (Gradient-weighted Class Activation Mapping)** to highlight the regions of the image that contributed most to the prediction.

**Task:** Image Classification & Explainability
**Model:** `microsoft/resnet-50`
**Library:** `pytorch-grad-cam`

## License

This notebook is licensed under the Apache License 2.0. See the [LICENSE](https://github.com/huggingface/transformers/blob/main/LICENSE) file for more details.

This notebook focuses on open-source, lightweight, and updated models, and is optimized to run on the Google Colab Free Tier.

## 1. Installation

In [ ]:
!pip install -q grad-cam transformers torch pillow numpy matplotlib

## 2. Imports

In [ ]:
import torch
import numpy as np
from PIL import Image
import requests
from transformers import AutoModelForImageClassification, AutoImageProcessor
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

## 3. Load Data

In [ ]:
# Load a standard image (Golden Retriever)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)
image = image.resize((224, 224))
print("Image loaded.")

## 4. Data Cleaning

In [ ]:
# Preprocess for the model
# Note: transformers ImageProcessor handles normalization, but Grad-CAM needs a normalized float tensor.
# We will normalize manually for the visualization consistency if needed, or extract from processor.

model_id = "microsoft/resnet-50"
processor = AutoImageProcessor.from_pretrained(model_id)

# Prepare input tensor
inputs = processor(images=image, return_tensors="pt")
input_tensor = inputs['pixel_values']
print("Data preprocessed.")

## 5. Load Model

In [ ]:
model = AutoModelForImageClassification.from_pretrained(model_id)
model.eval()
print("Model loaded.")

## 6. Inference

In [ ]:
# 1. Standard Inference
outputs = model(input_tensor)
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
label = model.config.id2label[predicted_class_idx]
print(f"Predicted Class: {label}")

# 2. Explainability (Grad-CAM)
# Identify target layer: For HF ResNet, it's usually the last stage
target_layers = [model.resnet.encoder.stages[-1].layers[-1]]

# Construct CAM object
# We need a custom reshape transform for some HF models, but ResNet is standard CNN.
cam = GradCAM(model=model, target_layers=target_layers)

# Generate CAM
targets = [ClassifierOutputTarget(predicted_class_idx)]
grayscale_cam = cam(input_tensor=input_tensor, targets=targets)

# Overlay on image
grayscale_cam = grayscale_cam[0, :]
rgb_img = np.float32(image) / 255
visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

# Convert back to PIL for display/saving
result_image = Image.fromarray(visualization)
print("Grad-CAM generated.")

## 7. Save Results

In [ ]:
# Save the visualization
image = result_image # Assign to 'image' for Kitchen Sink
text = f"Prediction: {label}"

# Kitchen Sink will run via update script
if hasattr(image, 'save'):
    image.save('generated_image.png')

# Kitchen Sink: Save Results
import os
import numpy as np
try:
    from PIL import Image
except ImportError:
    pass
try:
    import scipy.io.wavfile as wavfile
except ImportError:
    pass

print("Saving results...")

# 1. Text / Data / Variables
with open('results.txt', 'w') as f:
    if 'results' in locals():
        f.write(str(results))
    elif 'predictions' in locals():
        f.write(str(predictions))
    elif 'prediction' in locals():
        f.write(str(prediction))
    elif 'output' in locals():
        f.write(str(output))
    elif 'features' in locals():
        f.write(str(features))
    elif 'summary' in locals():
        f.write(str(summary))
    elif 'translation' in locals():
        f.write(str(translation))
    elif 'answers' in locals():
        f.write(str(answers))
    elif 'conversation' in locals():
        f.write(str(conversation))
    elif 'generated_text' in locals():
        f.write(str(generated_text))
    elif 'transcription' in locals():
        f.write(str(transcription))
    elif 'text' in locals():
        f.write(str(text))
    else:
        f.write("Task completed.")
print("Saved results.txt")

# 2. Images
if 'image' in locals():
    if hasattr(image, 'save'):
        image.save('generated_image.png')
        print("Saved generated_image.png")
elif 'images' in locals() and isinstance(images, list) and len(images) > 0:
    if hasattr(images[0], 'save'):
        images[0].save('generated_image.png')
        print("Saved generated_image.png (first image)")

# 3. Audio
if 'audio' in locals():
    if isinstance(audio, tuple):
        rate, data = audio
        try:
            wavfile.write('generated_audio.wav', rate, data)
            print("Saved generated_audio.wav")
        except:
            pass
    elif hasattr(audio, 'ndim'): # numpy array
        rate = save_rate if 'save_rate' in locals() else 16000
        try:
            wavfile.write('generated_audio.wav', rate, audio)
            print("Saved generated_audio.wav")
        except:
            pass
elif 'audio_data' in locals():
     rate = save_rate if 'save_rate' in locals() else 16000
     try:
         wavfile.write('generated_audio.wav', rate, audio_data)
         print("Saved generated_audio.wav")
     except:
         pass


## 8. Advanced Usage

**Teaching Tip:** Try varying the input parameters or prompts to see how the model's output changes. This helps build intuition about the model's capabilities and limitations.

Try targeting different layers (e.g., earlier layers) to see how features are built up. Or try a different model architecture like ViT (requires specific reshaping for Grad-CAM).